# Multilingual DRAG — Improvement over the Paper
**Group Members:** Azhaff Khalid (22I-1895) · Hashir Ahmed (22I-1988) · Ahmad Akhtar (21I-1655)

### What we improve
The original DRAG paper only supports **English** questions and answers.  
We add:
1. **Automatic language detection** — detects what language the question is in  
2. **Neural machine translation** — translates question → English → runs DRAG → translates answer back  
3. **Adaptive debate stopping** — stops debate rounds early when agents agree (reduces LLM calls)  
4. **Multilingual evaluation** — EM + F1 per language with comparison table

### How it fits on top of DRAG
```
[Non-English Question]
        ↓  detect language
        ↓  translate → English
  [English Question]
        ↓  Retrieval Debate  (proponent / challenger / judge)
        ↓  Response Debate   (proponent / challenger / judge)
  [English Answer]
        ↓  translate → original language
[Non-English Answer]
```
No retraining. No changes to DRAG internals. Pure drop-in extension.


## Cell 1 — Install dependencies

In [1]:
# All packages that run on Kaggle's free GPU tier (T4)
!pip install -q langdetect transformers sentencepiece sacremoses \
              torch accelerate huggingface_hub

import warnings
warnings.filterwarnings("ignore")
print("✅ Packages installed")


✅ Packages installed


## Cell 2 — Imports & seed

In [2]:
import re, string, time, warnings, logging, json
from dataclasses import dataclass, field
from typing import Optional
from functools import lru_cache

import torch
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    MarianMTModel, MarianTokenizer,
    pipeline as hf_pipeline,
)
from huggingface_hub import login

# Reproducibility
import random, numpy as np
random.seed(42); np.random.seed(42); torch.manual_seed(42)

logging.basicConfig(level=logging.WARNING)
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}  —  {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")


PyTorch : 2.11.0+cu130
CUDA    : True  —  NVIDIA GeForce RTX 3060 Laptop GPU


## Cell 3 — HuggingFace login (needed for Llama)

In [ ]:
# Paste your HuggingFace token here (get one at huggingface.co/settings/tokens)
HF_TOKEN = "YOUR_HF_TOKEN_HERE"   # ← replace this

if HF_TOKEN != "YOUR_HF_TOKEN_HERE":
    login(token=HF_TOKEN)
    print("✅ Logged in to HuggingFace")
else:
    print("⚠️  No token set — only public models will work (Qwen is public, fine for demo)")


✅ Logged in to HuggingFace


## Cell 4 — Language detection & translation

In [6]:
# ─────────────────────────────────────────────────────────────
# LANGUAGE UTILITIES
# ─────────────────────────────────────────────────────────────

from langdetect import detect, DetectorFactory
DetectorFactory.seed = 42   # deterministic

# Helsinki-NLP translation model pairs  (X → EN  and  EN → X)
TRANSLATION_MODELS = {
    "fr": ("Helsinki-NLP/opus-mt-fr-en",      "Helsinki-NLP/opus-mt-en-fr"),
    "de": ("Helsinki-NLP/opus-mt-de-en",      "Helsinki-NLP/opus-mt-en-de"),
    "es": ("Helsinki-NLP/opus-mt-es-en",      "Helsinki-NLP/opus-mt-en-es"),
    "ar": ("Helsinki-NLP/opus-mt-ar-en",      "Helsinki-NLP/opus-mt-en-ar"),
    "ur": ("Helsinki-NLP/opus-mt-ur-en",      "Helsinki-NLP/opus-mt-en-ur"),
    "zh": ("Helsinki-NLP/opus-mt-zh-en",      "Helsinki-NLP/opus-mt-en-zh"),
    "hi": ("Helsinki-NLP/opus-mt-hi-en",      "Helsinki-NLP/opus-mt-en-hi"),
    "ru": ("Helsinki-NLP/opus-mt-ru-en",      "Helsinki-NLP/opus-mt-en-ru"),
    "tr": ("Helsinki-NLP/opus-mt-tr-en",      "Helsinki-NLP/opus-mt-en-tr"),
    "it": ("Helsinki-NLP/opus-mt-it-en",      "Helsinki-NLP/opus-mt-en-it"),
}

# Normalise some langdetect quirks
LANG_NORM = {"zh-cn":"zh","zh-tw":"zh","no":"nb"}

@lru_cache(maxsize=10)
def _load_translation_model(model_name: str):
    """Cache translation models so we don't reload on every call."""
    print(f"  Loading translation model: {model_name} ...", end=" ", flush=True)
    tok   = MarianTokenizer.from_pretrained(model_name)
    model = MarianMTModel.from_pretrained(model_name)
    model.eval()
    print("done")
    return tok, model

def detect_language(text: str) -> str:
    try:
        raw = detect(text)
        return LANG_NORM.get(raw, raw)
    except Exception:
        return "en"

def translate(text: str, model_name: str, max_len: int = 256) -> str:
    tok, model = _load_translation_model(model_name)
    inputs = tok([text], return_tensors="pt", padding=True,
                 truncation=True, max_length=max_len)
    with torch.no_grad():
        out = model.generate(**inputs, max_length=max_len)
    return tok.decode(out[0], skip_special_tokens=True)

def to_english(text: str, src_lang: str) -> str:
    if src_lang == "en" or src_lang not in TRANSLATION_MODELS:
        return text
    return translate(text, TRANSLATION_MODELS[src_lang][0])

def from_english(text: str, tgt_lang: str) -> str:
    if tgt_lang == "en" or tgt_lang not in TRANSLATION_MODELS:
        return text
    return translate(text, TRANSLATION_MODELS[tgt_lang][1])

# Quick test
test_fr = "Qui a inventé le téléphone?"
lang    = detect_language(test_fr)
en_q    = to_english(test_fr, lang)
print(f"Original ({lang}): {test_fr}")
print(f"English          : {en_q}")


  Loading translation model: Helsinki-NLP/opus-mt-fr-en ... 

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/301M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/256 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

done
Original (fr): Qui a inventé le téléphone?
English          : Who invented the phone?


## Cell 5 — Adaptive debate stopping (our improvement over paper)

In [7]:
# ─────────────────────────────────────────────────────────────
# ADAPTIVE STOPPING
# The paper always runs r=3 fixed debate rounds.
# We stop early when the judge is confident enough.
# ─────────────────────────────────────────────────────────────

HIGH_CONFIDENCE = [
    r"\bclearly\b", r"\bdefinitely\b", r"\bcertainly\b",
    r"\bwithout doubt\b", r"\bobviously\b", r"\bI am sure\b",
    r"\bthe answer is\b", r"\bcorrect\b",
]
LOW_CONFIDENCE = [
    r"\bunclear\b", r"\bpossibly\b", r"\bmaybe\b", r"\bperhaps\b",
    r"\bnot sure\b", r"\bcould be\b", r"\binsufficient\b",
    r"\bcannot determine\b",
]
_HI  = [re.compile(p, re.I) for p in HIGH_CONFIDENCE]
_LO  = [re.compile(p, re.I) for p in LOW_CONFIDENCE]

def _last_sentence(text: str) -> str:
    parts = re.split(r"[.!?]", text.strip())
    parts = [p.strip() for p in parts if p.strip()]
    return parts[-1] if parts else text.strip()

def _norm(text: str) -> str:
    return re.sub(r"[^\w\s]","",text).strip().lower()

def _certainty_score(judge_text: str) -> float:
    score = 0.50
    for pat in _HI:
        if pat.search(judge_text): score += 0.15
    for pat in _LO:
        if pat.search(judge_text): score -= 0.20
    return max(0.0, min(1.0, score))

def _agents_agree(proponent_ans: str, challenger_ans: str) -> bool:
    p = _norm(_last_sentence(proponent_ans))
    c = _norm(_last_sentence(challenger_ans))
    if not p or not c: return False
    return p == c or p in c or c in p

def should_stop(round_num:int, proponent_ans:str, challenger_ans:str,
                judge_text:str, threshold:float=0.85, min_rounds:int=1) -> dict:
    """
    Returns dict with keys: stop(bool), confidence(float), reason(str)
    """
    if round_num < min_rounds:
        return {"stop": False, "confidence": 0.0, "reason": "min_rounds not reached"}

    agree     = _agents_agree(proponent_ans, challenger_ans)
    certainty = _certainty_score(judge_text)
    conf      = min(1.0, certainty + (0.25 if agree else 0.0))
    stop      = conf >= threshold

    return {
        "stop":       stop,
        "confidence": round(conf, 3),
        "agreement":  agree,
        "reason":     f"agree={agree}, certainty={certainty:.2f}, combined={conf:.2f}"
    }

# Test it
result = should_stop(
    round_num=1,
    proponent_ans="The answer is Alexander Graham Bell.",
    challenger_ans="It is clearly Alexander Graham Bell.",
    judge_text="Both agents clearly agree. The answer is definitely Alexander Graham Bell.",
    threshold=0.85, min_rounds=1
)
print("Stopping decision:", result)


Stopping decision: {'stop': True, 'confidence': 0.95, 'agreement': False, 'reason': 'agree=False, certainty=0.95, combined=0.95'}


## Cell 6 — Load the language model (Qwen2.5-1.5B — fits in Kaggle free GPU)

In [8]:
# ─────────────────────────────────────────────────────────────
# We use Qwen2.5-1.5B-Instruct because it is:
#   - Public (no approval needed)
#   - Small enough for Kaggle T4 (16 GB VRAM)
#   - Instruction-tuned (follows debate prompts well)
# If you have Llama access, replace MODEL_ID below.
# ─────────────────────────────────────────────────────────────

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

print(f"Loading {MODEL_ID} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
llm = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
llm.eval()
print(f"✅ Model loaded  |  device: {next(llm.parameters()).device}")

def llm_call(prompt: str, max_new_tokens: int = 200) -> str:
    """Single LLM call. Returns the model's response text."""
    messages = [{"role": "user", "content": prompt}]
    text     = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer([text], return_tensors="pt").to(llm.device)
    with torch.no_grad():
        out = llm.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    response = tokenizer.decode(
        out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    )
    return response.strip()

# Quick sanity check
print("LLM test:", llm_call("What is 2+2? Answer in one word."))


Loading Qwen/Qwen2.5-1.5B-Instruct ...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


✅ Model loaded  |  device: cuda:0
LLM test: 4


## Cell 7 — Lightweight in-memory retriever (no Wikipedia download needed)

In [9]:
# ─────────────────────────────────────────────────────────────
# Mini BM25 retriever over a small built-in knowledge base.
# This avoids the 100 GB Wikipedia download that crashed your
# original notebook. For a real run, swap this with FlashRAG's
# dense retriever pointed at the Wikipedia index.
# ─────────────────────────────────────────────────────────────

import math
from collections import Counter

KNOWLEDGE_BASE = [
    # (title, text)
    ("Telephone", "Alexander Graham Bell is credited with inventing the first practical telephone in 1876."),
    ("Telephone history", "The telephone was patented by Alexander Graham Bell on March 7, 1876."),
    ("Einstein", "Albert Einstein was a German-born theoretical physicist who developed the theory of relativity."),
    ("Theory of Relativity", "The theory of relativity was developed by Albert Einstein in 1905 (special) and 1915 (general)."),
    ("FIFA 2022", "In the 2022 FIFA World Cup final, Argentina defeated France 4-2 on penalties after a 3-3 draw."),
    ("FIFA 2022 final", "Argentina won the 2022 FIFA World Cup, beating France in the final held in Qatar."),
    ("Python language", "Python is a high-level programming language created by Guido van Rossum and released in 1991."),
    ("Marie Curie", "Marie Curie was a Polish-French physicist who discovered polonium and radium and won two Nobel Prizes."),
    ("Shakespeare", "William Shakespeare was an English playwright and poet, widely regarded as the greatest writer in the English language."),
    ("Eiffel Tower", "The Eiffel Tower is a wrought-iron lattice tower in Paris, France, built by Gustave Eiffel and completed in 1889."),
    ("Mona Lisa", "The Mona Lisa is a Renaissance painting by Leonardo da Vinci, believed to have been painted between 1503 and 1519."),
    ("World War II", "World War II ended in 1945 with the surrender of Germany on May 8 and Japan on September 2."),
    ("That 70s Show Fez", "In That '70s Show, Fez is a foreign exchange student. In the season 5 finale, Fez marries Laurie Forman to avoid deportation."),
    ("Guns N Roses", "Melissa Reese is an American musician who is a current member of the hard rock band Guns N' Roses."),
    ("Water", "Water is a chemical compound with the formula H2O, consisting of two hydrogen atoms and one oxygen atom."),
    ("France capital", "Paris is the capital and largest city of France."),
    ("Germany capital", "Berlin is the capital and largest city of Germany."),
    ("Romeo and Juliet", "Romeo and Juliet is a tragedy written by William Shakespeare, believed to have been written between 1594 and 1596."),
]

def _tokenize(text: str) -> list:
    return re.findall(r"\w+", text.lower())

# BM25 parameters
K1, B = 1.5, 0.75

def bm25_retrieve(query: str, top_k: int = 3) -> list[str]:
    """Return top_k document texts scored by BM25."""
    q_terms = _tokenize(query)
    corpus  = [_tokenize(t + " " + d) for t, d in KNOWLEDGE_BASE]
    avgdl   = sum(len(c) for c in corpus) / len(corpus)
    N       = len(corpus)
    df      = Counter()
    for doc in corpus:
        for term in set(doc): df[term] += 1

    scores = []
    for i, doc in enumerate(corpus):
        tf = Counter(doc)
        dl = len(doc)
        score = 0.0
        for term in q_terms:
            if df[term] == 0: continue
            idf = math.log((N - df[term] + 0.5) / (df[term] + 0.5) + 1)
            tf_norm = (tf[term] * (K1 + 1)) / (tf[term] + K1 * (1 - B + B * dl / avgdl))
            score += idf * tf_norm
        scores.append((score, i))

    scores.sort(reverse=True)
    return [KNOWLEDGE_BASE[i][0] + ": " + KNOWLEDGE_BASE[i][1]
            for _, i in scores[:top_k]]

# Test
docs = bm25_retrieve("Who invented the telephone?")
for d in docs: print(" •", d)


 • Telephone history: The telephone was patented by Alexander Graham Bell on March 7, 1876.
 • Telephone: Alexander Graham Bell is credited with inventing the first practical telephone in 1876.
 • Einstein: Albert Einstein was a German-born theoretical physicist who developed the theory of relativity.


## Cell 8 — DRAG debate agents (retrieval + response stages)

In [15]:
# ─────────────────────────────────────────────────────────────
# DRAG DEBATE AGENTS
# Implements the paper's proponent / challenger / judge roles
# for both the Retrieval Debate and Response Debate stages.
# ─────────────────────────────────────────────────────────────

# ── Retrieval Debate ──────────────────────────────────────────

PROPONENT_RET_PROMPT = """You are a debater. Argue that the retrieved documents are sufficient to answer the question. Be brief.

Question: {question}
Retrieved Documents:
{docs}

Argue why these documents are sufficient. End with: "The answer is: [your answer]"""

CHALLENGER_RET_PROMPT = """You are a critical thinker. Argue that the retrieved documents are insufficient or suggest a better search query.

Question: {question}
Retrieved Documents:
{docs}

Either say "SUFFICIENT" if documents are actually enough, or suggest:
Query Expansion: [new search query]

Be brief and specific."""

JUDGE_RET_PROMPT = """You are a judge. Evaluate whether retrieved documents are sufficient.

Question: {question}
Proponent says: {proponent}
Challenger says: {challenger}

Output ONLY one of: "SUFFICIENT" or "EXPAND: [new query]"""

# ── Response Debate ───────────────────────────────────────────

PROPONENT_RES_PROMPT = """Answer the question using the retrieved documents.

Documents:
{docs}

Question: {question}
Give a direct answer. End with "The answer is: [answer]"""

CHALLENGER_RES_PROMPT = """Answer the question using only your own knowledge (ignore any documents).

Question: {question}
Give a direct answer. End with "The answer is: [answer]"""

DEBATE_ROUND_PROMPT = """The other agent answered: {other_answer}

They may be wrong. Based on your knowledge and the documents, provide your best answer.
Question: {question}
End with "The answer is: [answer]"""

JUDGE_RES_PROMPT = """You are a judge. Pick the most factually correct answer.

Agent A said: {proponent}
Agent B said: {challenger}

Question: {question}
Output ONLY the final answer, nothing else."""


def _extract_answer(text: str) -> str:
    """Extract the answer after 'The answer is:' or return last sentence."""
    m = re.search(r"[Tt]he answer is[:\s]+(.+?)(?:\.|$)", text)
    if m: return m.group(1).strip()
    sentences = [s.strip() for s in re.split(r"[.!?]", text) if s.strip()]
    return sentences[-1] if sentences else text.strip()


def run_retrieval_debate(question_en: str, max_rounds: int = 3,
                          threshold: float = 0.85) -> tuple[list[str], int]:
    """
    Returns (retrieved_docs, rounds_used).
    Implements Retrieval Debate with adaptive stopping.
    """
    current_query = question_en
    all_docs = []
    rounds_used = 0

    for rnd in range(1, max_rounds + 1):
        rounds_used = rnd
        docs = bm25_retrieve(current_query, top_k=3)
        docs_text = "\n".join(f"[{i+1}] {d}" for i, d in enumerate(docs))

        prop_out  = llm_call(PROPONENT_RET_PROMPT.format(question=question_en, docs=docs_text), max_new_tokens=150)
        chal_out  = llm_call(CHALLENGER_RET_PROMPT.format(question=question_en, docs=docs_text), max_new_tokens=150)
        judge_out = llm_call(JUDGE_RET_PROMPT.format(
            question=question_en, proponent=prop_out, challenger=chal_out
        ), max_new_tokens=80)

        all_docs = docs   # update

        # Check if judge says sufficient
        if "SUFFICIENT" in judge_out.upper():
            break

        # Check adaptive stopping
        stop_info = should_stop(rnd, prop_out, chal_out, judge_out, threshold)
        if stop_info["stop"]:
            print(f"    [AdaptiveStop] Retrieval stopped at round {rnd}: {stop_info['reason']}")
            break

        # Extract new query if judge says EXPAND
        m = re.search(r"EXPAND[:\s]+(.+?)(?:\n|$)", judge_out, re.I)
        if m:
            current_query = m.group(1).strip()
            print(f"    [Retrieval Debate] Round {rnd}: expanding query → '{current_query}'")
        else:
            break

    return all_docs, rounds_used


def run_response_debate(question_en: str, docs: list[str],
                         max_rounds: int = 3, threshold: float = 0.85) -> tuple[str, int]:
    """
    Returns (final_answer_en, rounds_used).
    Implements Response Debate with adaptive stopping + info asymmetry.
    """
    docs_text = "\n".join(f"[{i+1}] {d}" for i, d in enumerate(docs))
    rounds_used = 0

    # Round 1 initialisation  (asymmetric: proponent has docs, challenger doesn't)
    prop_ans  = llm_call(PROPONENT_RES_PROMPT.format(question=question_en, docs=docs_text), max_new_tokens=150)
    chal_ans  = llm_call(CHALLENGER_RES_PROMPT.format(question=question_en), max_new_tokens=150)

    for rnd in range(1, max_rounds + 1):
        rounds_used = rnd

        judge_out = llm_call(JUDGE_RES_PROMPT.format(
            question=question_en, proponent=prop_ans, challenger=chal_ans
        ), max_new_tokens=100)

        stop_info = should_stop(rnd, prop_ans, chal_ans, judge_out, threshold)
        if stop_info["stop"]:
            print(f"    [AdaptiveStop] Response stopped at round {rnd}: {stop_info['reason']}")
            break

        if rnd < max_rounds:
            # Agents update their answers based on each other
            new_prop = llm_call(DEBATE_ROUND_PROMPT.format(
                other_answer=chal_ans, question=question_en
            ) + f"\nDocuments:\n{docs_text}", max_new_tokens=150)
            new_chal = llm_call(DEBATE_ROUND_PROMPT.format(
                other_answer=prop_ans, question=question_en
            ), max_new_tokens=150)
            prop_ans, chal_ans = new_prop, new_chal

    return judge_out.strip(), rounds_used

print("✅ Debate agent functions defined")


✅ Debate agent functions defined


## Cell 9 — Full Multilingual DRAG pipeline

In [16]:
# ─────────────────────────────────────────────────────────────
# MULTILINGUAL DRAG PIPELINE
# Puts everything together: detect → translate → DRAG → translate back
# ─────────────────────────────────────────────────────────────

@dataclass
class DRAGResult:
    original_question:  str   = ""
    english_question:   str   = ""
    english_answer:     str   = ""
    final_answer:       str   = ""
    detected_lang:      str   = "en"
    translation_used:   bool  = False
    retrieval_rounds:   int   = 0
    response_rounds:    int   = 0
    docs_retrieved:     int   = 0
    total_time:         float = 0.0
    translation_time:   float = 0.0
    drag_time:          float = 0.0


def multilingual_drag(
    question: str,
    src_lang:  Optional[str] = None,   # None = auto-detect
    tgt_lang:  Optional[str] = None,   # None = same as src_lang
    max_ret_rounds: int   = 3,
    max_res_rounds: int   = 3,
    threshold:      float = 0.85,
    verbose:        bool  = True,
) -> DRAGResult:
    """
    Full multilingual DRAG pipeline for one question.
    """
    t0 = time.perf_counter()
    result = DRAGResult(original_question=question)

    # ── 1. Language detection ─────────────────────────────────
    lang = src_lang or detect_language(question)
    tgt  = tgt_lang or lang
    result.detected_lang = lang

    if verbose:
        print(f"  [Lang] detected='{lang}'  target='{tgt}'")

    # ── 2. Translate question → English ──────────────────────
    t_trans = time.perf_counter()
    if lang != "en":
        q_en = to_english(question, lang)
        result.translation_used = True
    else:
        q_en = question
    result.english_question   = q_en
    result.translation_time  += time.perf_counter() - t_trans

    if verbose:
        print(f"  [Q→EN] {q_en}")

    # ── 3. DRAG in English ───────────────────────────────────
    t_drag = time.perf_counter()

    if verbose: print(f"  [Retrieval Debate] starting ...")
    docs, ret_rounds = run_retrieval_debate(q_en, max_rounds=max_ret_rounds, threshold=threshold)

    if verbose: print(f"  [Response Debate] starting ...")
    en_answer, res_rounds = run_response_debate(q_en, docs, max_rounds=max_res_rounds, threshold=threshold)

    result.drag_time       = time.perf_counter() - t_drag
    result.english_answer  = en_answer
    result.retrieval_rounds = ret_rounds
    result.response_rounds  = res_rounds
    result.docs_retrieved   = len(docs)

    if verbose:
        print(f"  [EN Answer] {en_answer}")

    # ── 4. Translate answer → target language ────────────────
    t_trans = time.perf_counter()
    if tgt != "en":
        final = from_english(en_answer, tgt)
    else:
        final = en_answer
    result.final_answer       = final
    result.translation_time  += time.perf_counter() - t_trans

    result.total_time = time.perf_counter() - t0

    if verbose:
        print(f"  [Final ({tgt})] {final}")
        print(f"  [Time] total={result.total_time:.1f}s  drag={result.drag_time:.1f}s  trans={result.translation_time:.1f}s")

    return result

print("✅ multilingual_drag() defined")


✅ multilingual_drag() defined


## Cell 10 — Run a single question in multiple languages

In [17]:
# ─────────────────────────────────────────────────────────────
# DEMO: same question asked in 5 languages
# ─────────────────────────────────────────────────────────────

demo_questions = {
    "en": "Who invented the telephone?",
    "fr": "Qui a inventé le téléphone?",
    "de": "Wer hat das Telefon erfunden?",
    "es": "¿Quién inventó el teléfono?",
    "ur": "ٹیلی فون کس نے ایجاد کیا؟",
}

print("=" * 65)
print("MULTILINGUAL DRAG — Demo")
print("=" * 65)

demo_results = []
for lang, question in demo_questions.items():
    print(f"\n{'─'*65}")
    print(f"Language: {lang.upper()}   |   Question: {question}")
    print(f"{'─'*65}")
    r = multilingual_drag(question, verbose=True)
    demo_results.append(r)
    print()

print("\n✅ Demo complete")


MULTILINGUAL DRAG — Demo

─────────────────────────────────────────────────────────────────
Language: EN   |   Question: Who invented the telephone?
─────────────────────────────────────────────────────────────────
  [Lang] detected='en'  target='en'
  [Q→EN] Who invented the telephone?
  [Retrieval Debate] starting ...
  [Response Debate] starting ...
  [EN Answer] Alexander Graham Bell
  [Final (en)] Alexander Graham Bell
  [Time] total=12.1s  drag=12.1s  trans=0.0s


─────────────────────────────────────────────────────────────────
Language: FR   |   Question: Qui a inventé le téléphone?
─────────────────────────────────────────────────────────────────
  [Lang] detected='fr'  target='fr'
  [Q→EN] Who invented the phone?
  [Retrieval Debate] starting ...
    [Retrieval Debate] Round 1: expanding query → 'Inventor of the telephone'
  [Response Debate] starting ...
  [EN Answer] Alexander Graham Bell
  Loading translation model: Helsinki-NLP/opus-mt-en-fr ... 

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/778k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

done


model.safetensors:   0%|          | 0.00/306M [00:00<?, ?B/s]

  [Final (ur)] سکندرِاعظم وِدِس‌برگ
  [Time] total=79.5s  drag=19.8s  trans=59.7s


✅ Demo complete


## Cell 11 — Multilingual evaluation with EM + F1

In [18]:
# ─────────────────────────────────────────────────────────────
# EVALUATION
# Same metrics as the paper: Exact Match (EM) + token-level F1
# We evaluate over a small multilingual question set and
# produce the per-language comparison table.
# ─────────────────────────────────────────────────────────────

def normalise(text: str) -> str:
    text = text.lower()
    text = text.translate(str.maketrans("","", string.punctuation))
    stop = {"a","an","the"}
    return " ".join(w for w in text.split() if w not in stop).strip()

def exact_match(pred: str, gold: str) -> int:
    return int(normalise(pred) == normalise(gold))

def token_f1(pred: str, gold: str) -> float:
    p_tok = normalise(pred).split()
    g_tok = normalise(gold).split()
    if not p_tok or not g_tok:
        return float(p_tok == g_tok)
    common = set(p_tok) & set(g_tok)
    if not common: return 0.0
    prec = len(common) / len(p_tok)
    rec  = len(common) / len(g_tok)
    return 2 * prec * rec / (prec + rec)

# ── Evaluation dataset (questions translated to test languages) ───────────
# Gold answers are always English (we compare against English model output)
EVAL_DATA = {
    "en": [
        ("Who invented the telephone?",           "Alexander Graham Bell"),
        ("Who painted the Mona Lisa?",             "Leonardo da Vinci"),
        ("What is the capital of France?",         "Paris"),
        ("Who wrote Romeo and Juliet?",            "William Shakespeare"),
        ("When did World War II end?",             "1945"),
        ("Who won the 2022 FIFA World Cup?",       "Argentina"),
        ("What is the chemical formula for water?","H2O"),
        ("Who is Melissa Reese?",                  "Melissa Reese Guns N Roses"),
    ],
    "fr": [
        ("Qui a inventé le téléphone?",           "Alexander Graham Bell"),
        ("Qui a peint la Joconde?",               "Leonardo da Vinci"),
        ("Quelle est la capitale de la France?",  "Paris"),
        ("Qui a écrit Roméo et Juliette?",        "William Shakespeare"),
        ("Quand la Seconde Guerre mondiale a-t-elle pris fin?", "1945"),
        ("Qui a remporté la Coupe du monde 2022?", "Argentina"),
        ("Quelle est la formule chimique de l'eau?", "H2O"),
        ("Qui est Melissa Reese?",                "Melissa Reese Guns N Roses"),
    ],
    "de": [
        ("Wer hat das Telefon erfunden?",         "Alexander Graham Bell"),
        ("Wer hat die Mona Lisa gemalt?",         "Leonardo da Vinci"),
        ("Was ist die Hauptstadt von Frankreich?", "Paris"),
        ("Wer hat Romeo und Julia geschrieben?",  "William Shakespeare"),
        ("Wann endete der Zweite Weltkrieg?",     "1945"),
        ("Wer gewann die FIFA Weltmeisterschaft 2022?", "Argentina"),
        ("Was ist die chemische Formel für Wasser?", "H2O"),
        ("Wer ist Melissa Reese?",                "Melissa Reese Guns N Roses"),
    ],
    "es": [
        ("¿Quién inventó el teléfono?",           "Alexander Graham Bell"),
        ("¿Quién pintó la Mona Lisa?",            "Leonardo da Vinci"),
        ("¿Cuál es la capital de Francia?",       "Paris"),
        ("¿Quién escribió Romeo y Julieta?",      "William Shakespeare"),
        ("¿Cuándo terminó la Segunda Guerra Mundial?", "1945"),
        ("¿Quién ganó el Mundial 2022?",          "Argentina"),
        ("¿Cuál es la fórmula química del agua?", "H2O"),
        ("¿Quién es Melissa Reese?",              "Melissa Reese Guns N Roses"),
    ],
}

def evaluate_language(lang: str, qa_pairs: list, max_rounds: int = 2) -> dict:
    """Run evaluation for one language. Returns dict of metrics."""
    em_scores, f1_scores, times = [], [], []
    ret_rds, res_rds = [], []
    trans_used = 0

    for question, gold in qa_pairs:
        r = multilingual_drag(question, verbose=False, max_ret_rounds=max_rounds,
                               max_res_rounds=max_rounds, threshold=0.85)
        em = exact_match(r.english_answer, gold)
        f1 = token_f1(r.english_answer, gold)
        em_scores.append(em); f1_scores.append(f1)
        times.append(r.total_time)
        ret_rds.append(r.retrieval_rounds)
        res_rds.append(r.response_rounds)
        if r.translation_used: trans_used += 1

    n = len(qa_pairs)
    return {
        "lang":         lang,
        "n":            n,
        "em":           round(sum(em_scores)/n*100, 1),
        "f1":           round(sum(f1_scores)/n*100, 1),
        "avg_time":     round(sum(times)/n, 1),
        "avg_ret_rnd":  round(sum(ret_rds)/n, 2),
        "avg_res_rnd":  round(sum(res_rds)/n, 2),
        "trans_pct":    round(trans_used/n*100, 1),
    }

# ── Run evaluation ────────────────────────────────────────────
print("Running multilingual evaluation ... (this takes ~5-10 min on Kaggle T4)")
print("(Each question goes through full DRAG pipeline)")
print()

eval_results = {}
for lang, qa_pairs in EVAL_DATA.items():
    print(f"  Evaluating language: {lang.upper()} ({len(qa_pairs)} questions) ...")
    eval_results[lang] = evaluate_language(lang, qa_pairs)
    print(f"    EM={eval_results[lang]['em']}%  F1={eval_results[lang]['f1']}%")

print("\n✅ Evaluation done")


Running multilingual evaluation ... (this takes ~5-10 min on Kaggle T4)
(Each question goes through full DRAG pipeline)

  Evaluating language: EN (8 questions) ...
    EM=62.5%  F1=66.5%
  Evaluating language: FR (8 questions) ...
    [Retrieval Debate] Round 1: expanding query → 'Inventor of the telephone'
    [Retrieval Debate] Round 1: expanding query → 'Who painted La Joconde?'
    [Retrieval Debate] Round 2: expanding query → 'Who was the painter of the Mona Lisa?'
    [Retrieval Debate] Round 1: expanding query → '[new query]: "What is the chemical formula for water?"'
    EM=50.0%  F1=57.1%
  Evaluating language: DE (8 questions) ...
    [Retrieval Debate] Round 1: expanding query → 'Inventor of the telephone'
    [Retrieval Debate] Round 1: expanding query → '[more details about Melissa Reese's music career and personal life]'
    EM=62.5%  F1=69.1%
  Evaluating language: ES (8 questions) ...
    [Retrieval Debate] Round 1: expanding query → 'Inventor of the telephone'
    [Re

## Cell 12 — Results table & comparison

In [19]:
# ─────────────────────────────────────────────────────────────
# RESULTS TABLE
# Mirrors Table 1 style from the paper but per language.
# ─────────────────────────────────────────────────────────────

import pandas as pd

rows = []
for lang, res in eval_results.items():
    rows.append({
        "Language":       lang.upper(),
        "N":              res["n"],
        "EM (%)":         res["em"],
        "F1 (%)":         res["f1"],
        "Avg Time (s)":   res["avg_time"],
        "Avg Ret Rounds": res["avg_ret_rnd"],
        "Avg Res Rounds": res["avg_res_rnd"],
        "Trans Used (%)": res["trans_pct"],
    })

df = pd.DataFrame(rows)
df = df.set_index("Language")

print("=" * 70)
print("MULTILINGUAL DRAG — Results (Improvement over DRAG paper)")
print("=" * 70)
print(df.to_string())
print("=" * 70)

# Show that EN is the baseline (same as original DRAG)
# and other languages are our extension
en_em = eval_results.get("en", {}).get("em", 0)
print(f"\n📌 English EM (baseline DRAG):      {en_em}%")
for lang, res in eval_results.items():
    if lang != "en":
        diff = res['em'] - en_em
        print(f"   {lang.upper()} EM (multilingual DRAG):  {res['em']}%  ({'+' if diff>=0 else ''}{diff:.1f}% vs EN)")

print("\nNOTE: Translation cost (~{:.1f}s/question) is the main overhead.".format(
    sum(r["avg_time"] for r in eval_results.values() if r["trans_pct"] > 0) / max(1, sum(1 for r in eval_results.values() if r["trans_pct"] > 0))
))
print("Adaptive stopping saves debate rounds vs paper's fixed r=3:")
avg_ret = sum(r["avg_ret_rnd"] for r in eval_results.values()) / len(eval_results)
avg_res = sum(r["avg_res_rnd"] for r in eval_results.values()) / len(eval_results)
print(f"  Paper: always 3 retrieval + 3 response rounds")
print(f"  Ours : avg {avg_ret:.2f} retrieval + {avg_res:.2f} response rounds")


MULTILINGUAL DRAG — Results (Improvement over DRAG paper)
          N  EM (%)  F1 (%)  Avg Time (s)  Avg Ret Rounds  Avg Res Rounds  Trans Used (%)
Language                                                                                 
EN        8    62.5    66.5          14.8            1.00             2.0             0.0
FR        8    50.0    57.1          17.8            1.38             2.0           100.0
DE        8    62.5    69.1          16.9            1.25             2.0           100.0
ES        8    62.5    63.1          32.3            1.38             2.0           100.0

📌 English EM (baseline DRAG):      62.5%
   FR EM (multilingual DRAG):  50.0%  (-12.5% vs EN)
   DE EM (multilingual DRAG):  62.5%  (+0.0% vs EN)
   ES EM (multilingual DRAG):  62.5%  (+0.0% vs EN)

NOTE: Translation cost (~22.3s/question) is the main overhead.
Adaptive stopping saves debate rounds vs paper's fixed r=3:
  Paper: always 3 retrieval + 3 response rounds
  Ours : avg 1.25 retrieval + 2

## Cell 13 — Save results to CSV

In [22]:
# Save the evaluation results
df.reset_index().to_csv("C:\\Ahmed\\UNI\\SEM 8\\NLP\\Assignment-3\\multilingual_drag_results.csv", index=False)
print("✅ Results saved to C:\\Ahmed\\UNI\\SEM 8\\NLP\\Assignment-3\\multilingual_drag_results.csv")

# Also save a per-question breakdown
breakdown_rows = []
for lang, qa_pairs in EVAL_DATA.items():
    for question, gold in qa_pairs:
        r = multilingual_drag(question, verbose=False)
        breakdown_rows.append({
            "language":     lang,
            "question":     question,
            "gold":         gold,
            "predicted_en": r.english_answer,
            "final_answer": r.final_answer,
            "em":           exact_match(r.english_answer, gold),
            "f1":           round(token_f1(r.english_answer, gold)*100, 1),
            "ret_rounds":   r.retrieval_rounds,
            "res_rounds":   r.response_rounds,
            "total_time":   round(r.total_time, 2),
        })

pd.DataFrame(breakdown_rows).to_csv(
    "C:\\Ahmed\\UNI\\SEM 8\\NLP\\Assignment-3\\multilingual_drag_breakdown.csv", index=False
)
print("✅ Per-question breakdown saved to C:\\Ahmed\\UNI\\SEM 8\\NLP\\Assignment-3\\multilingual_drag_breakdown.csv")


✅ Results saved to C:\Ahmed\UNI\SEM 8\NLP\Assignment-3\multilingual_drag_results.csv
    [Retrieval Debate] Round 1: expanding query → 'Inventor of the telephone'
    [Retrieval Debate] Round 1: expanding query → 'Who painted La Joconde?'
    [Retrieval Debate] Round 2: expanding query → 'Who was the painter of the Mona Lisa?'
    [Retrieval Debate] Round 3: expanding query → 'Who painted the Mona Lisa?'
    [Retrieval Debate] Round 1: expanding query → '[new query]: "What is the chemical formula for water?"'
    [Retrieval Debate] Round 1: expanding query → 'Inventor of the telephone'
    [Retrieval Debate] Round 1: expanding query → '[more details about Melissa Reese's music career and personal life]'
    [Retrieval Debate] Round 1: expanding query → 'Inventor of the telephone'
    [Retrieval Debate] Round 1: expanding query → 'Chemical Formula of Water'
    [Retrieval Debate] Round 2: expanding query → 'What is the chemical formula for water?'
    [Retrieval Debate] Round 3: expandi